# CNN2 con data augmentation en PyTorch

Este notebook traduce la arquitectura `CNN2_aug` del Lab3 sin entrenar el dataset completo. El objetivo es comprobar arquitectura, flujo de datos, augmentation, dispositivo y un único paso de optimización con un batch de 32 imágenes.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from torch import nn
from torch.utils.data import DataLoader

from src.data.asl_dataset import ASLNpyDataset, CNN2Augmentation
from src.models.cnn2 import (
    create_cnn2, keras_compatible_parameter_count, make_adam,
    trainable_parameter_count, load_initial_checkpoint,
)
from src.utils.device import select_device

## 1. Arquitectura traducida

La entrada cambia de NHWC `(N,64,64,3)` a NCHW `(N,3,64,64)`. No se agregan capas.

| Bloque | PyTorch | Salida |
|---|---|---|
| 1 | Conv 3→32, 3×3, padding 1; ReLU; BN; MaxPool 2; Dropout 0.25 | 32×32×32 |
| 2 | Conv 32→64, 3×3, padding 1; ReLU; BN; MaxPool 2; Dropout 0.25 | 64×16×16 |
| 3 | Conv 64→128, 3×3, padding 1; ReLU; BN; MaxPool 2; Dropout 0.25 | 128×8×8 |
| Salida | AdaptiveAvgPool(1); Dropout 0.50; Linear 128→29 | 29 logits |

BatchNorm usa `eps=1e-3` y `momentum=0.01`: esta última cifra corresponde a la convención inversa del `momentum=0.99` de Keras. La red devuelve logits porque `CrossEntropyLoss` incorpora softmax.

In [ ]:
model = create_cnn2(seed=42)
print(model)
print('Parámetros entrenables PyTorch:', trainable_parameter_count(model))
print('Conteo compatible con Keras:', keras_compatible_parameter_count(model))
assert trainable_parameter_count(model) == 97_437
assert keras_compatible_parameter_count(model) == 97_885

### ¿Por qué PyTorch no muestra 97,885 `Parameter`?

Keras cuenta 97,437 valores entrenables y 448 medias/varianzas móviles de BatchNorm. PyTorch conserva esos 448 valores como *buffers*, no como `Parameter`. Además mantiene tres contadores `num_batches_tracked`, que son internos y no tienen equivalente Keras. El conteo comparable es, por tanto, **97,437 + 448 = 97,885**.

## 2. Datos y memoria

`ASLNpyDataset` abre los `.npy` con `mmap_mode='r'`. Cada `__getitem__` copia solamente una imagen, cambia NHWC→NCHW y normaliza `uint8 / 255` a `float32`. No crea una copia float32 completa de entrenamiento.

- `X_train.npy` uint8: ~142.73 MiB en disco.
- Copia completa float32 evitada: ~570.94 MiB.
- Batch float32 de 32 imágenes: 1.50 MiB, sin contar activaciones del modelo.

Validación y prueba usan exactamente esta conversión determinista y nunca reciben augmentation aleatorio.

In [ ]:
train_dataset = ASLNpyDataset(ROOT / 'data', 'train')
val_dataset = ASLNpyDataset(ROOT / 'data', 'val')
test_dataset = ASLNpyDataset(ROOT / 'data', 'test')
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False, num_workers=0)
images, labels = next(iter(train_loader))
print('Splits:', len(train_dataset), len(val_dataset), len(test_dataset))
print('Batch:', images.shape, images.dtype, float(images.min()), float(images.max()))
print('Labels:', labels.shape, labels.dtype, int(labels.min()), int(labels.max()))
print('Memoria:', train_dataset.memory_report(batch_size=32))

## 3. Augmentation equivalente

Orden conservado: rotación ±14.4°, traslación ±10%, zoom ±10%, brillo ±0.2 y contraste con factor `[0.8,1.2]`. No se usa flip horizontal. Las operaciones geométricas son bilineales con relleno reflect.

Diferencias declaradas:

1. TensorFlow y PyTorch usan RNG y convenciones de interpolación distintas; las distribuciones son equivalentes, pero una muestra aumentada no será idéntica píxel a píxel.
2. `grid_sample(..., align_corners=False)` aproxima la geometría de Keras, con posibles diferencias subpíxel en bordes.
3. Se corrige intencionalmente `RandomContrast`: el Lab3 dejó el rango predeterminado `[0,255]` aunque la entrada estaba en `[0,1]`; aquí el contraste y clipping usan `[0,1]`.
4. El mismo módulo tensorial se mueve a CPU o CUDA. No hay dos políticas de augmentation.

In [ ]:
device = select_device('cpu')  # Cambiar explícitamente a 'cuda' solo en la PC NVIDIA.
augmentation = CNN2Augmentation().to(device)
augmentation.train()
torch.manual_seed(42)
augmented = augmentation(images.to(device))
print(device, augmented.shape, float(augmented.min()), float(augmented.max()))
assert augmented.shape == images.shape
assert torch.isfinite(augmented).all()

augmentation.eval()
assert torch.equal(augmentation(images.to(device)), images.to(device))

## 4. Un paso de verificación, no entrenamiento

El siguiente código ejecuta forward, backward y un único `Adam.step()` sobre 32 imágenes. Adam conserva `lr=1e-3`, β=(0.9,0.999), ε=1e-7 y weight decay 0. No representa una época ni produce métricas del modelo.

In [ ]:
model = create_cnn2(seed=42).to(device)
optimizer = make_adam(model)
criterion = nn.CrossEntropyLoss()
model.train()
optimizer.zero_grad(set_to_none=True)
logits = model(images.to(device))
loss = criterion(logits, labels.to(device))
loss.backward()
optimizer.step()
print('Logits:', logits.shape, '| loss finita:', bool(torch.isfinite(loss)))
assert logits.shape == (32, 29)
assert torch.isfinite(logits).all() and torch.isfinite(loss)

## 5. Estado inicial canónico

Los benchmarks deben cargar `results/initialization/cnn2_seed42_initial.pt`, no volver a inicializar cada dispositivo. El checkpoint contiene seed, metadatos y `state_dict` creado en CPU. Así CPU y CUDA parten de los mismos bytes.

In [ ]:
checkpoint = ROOT / 'results' / 'initialization' / 'cnn2_seed42_initial.pt'
fresh_model = create_cnn2(seed=999)
metadata = load_initial_checkpoint(fresh_model, checkpoint)
print(metadata)
assert metadata['seed'] == 42
assert metadata['keras_compatible_parameters'] == 97_885

## 6. Validación automatizada

Desde la raíz:

```powershell
python -m unittest -v tests.test_model
python scripts/create_initial_state.py
```

La ejecución CPU realizada durante esta implementación pasó 9/9 pruebas. No se ejecutó entrenamiento completo ni prueba GPU.